# Scene-Graph Interfaces for π0.5 on RoboMME: Data to Training (Pilot, π0.5 version)

This is the π0.5 version of `sg_interfaces_pilot.ipynb` (v1, SmolVLA, kept as the record of the first pilot). Steps 1-7 (data, replay, oracle graph, perception, update rules, quality levels) are the same code through `sg_pilot.py`; Steps 8-10 use π0.5 through `pi05_policy.py`, with the RoboMME training recipe adapted to one 24 GB GPU (LoRA instead of full fine-tuning).

Run on a Linux machine with an NVIDIA GPU and working Vulkan: **Kernel → Restart & Run All**. `RUN_MODE` in the Setup cell: `"smoke"` (one task, a few training steps, 1-2 evaluation episodes) or `"pilot"` (the four permanence tasks, trained outside the notebook with `run_arms_parallel.sh --policy pi05`).

## At a Glance

This notebook tests how a robot policy should be handed an explicit memory of objects it can no longer see: it builds a scene graph of the objects in a RoboMME task, gives that graph to the SmolVLA policy through three different input routes, and runs the pilot that decides whether the full comparison is worth its compute. Results: = ? (filled after the pilot run).

### What this work is about

**The subject: delivering a memory to a policy.** Think of the shell game. A ball sits under one of three identical cups, and the cups are shuffled. Someone who watched the shuffle knows which cup to lift; someone who walks in afterwards sees three identical cups and can only guess. A helper who watched can tell the latecomer in three ways: point at the cup, say "the one on the right", or hand over a card in a format the latecomer was trained to read. Two questions follow. Which way does the latecomer use best? And which way does the least damage when the helper is sometimes wrong? In this work the helper is a **scene graph**, the latecomer is a **policy** that sees only the current camera frame, and the three ways are the three **interfaces** compared here. Where the analogy stops: the policy is not told what the hint means; it has to learn to use it during fine-tuning.

**The task, RoboMME VideoUnmaskSwap.** Before the robot moves, it is shown a video: a red, a green and a blue cube stand on the table, containers come down over them, and pairs of containers swap places one to three times. Then the robot is told, for example, "pick up the container that hides the red cube". All containers look alike, so the frame the robot sees at that moment does not contain the answer; only the video does (`robomme_env/VideoUnmaskSwap.py`).

**One scene-graph node.** A scene graph here is a table with one row per object and eight numbers per row: `[id, label, u, v, x, y, z, hidden]`, where (u, v) is the object's pixel in the front camera, (x, y, z) its position on the table in metres, and hidden says whether any camera sees it. A small worked case (illustrative: the synthetic scene used to test the code, not a RoboMME episode) has two containers that swap places while the red cube is hidden under the left one:

| Moment | Container 0 (u, v) | Container 1 (u, v) | Red cube, hidden | Red cube (u, v), kept by the rule that freezes it | Red cube (u, v), kept by the rule that moves it with its container |
|---|---|---|---|---|---|
| cube just covered | (100, 100) | (156, 100) | yes | (100, 103) | (100, 103) |
| after the swap | (156, 100) | (100, 100) | yes | (100, 103), now under the wrong container | (155, 103), under container 0 |

The first rule is the original paper's: a hidden object keeps its last seen position. The second, called **bound** here, attaches a hidden object to the container it disappeared under and moves it with that container; the simulator moves cubes exactly this way during the swap. In the code test, the frozen position ends 53 px (0.20 m) from the truth and the bound position ends 0 px from it.

**The three interfaces, on that one row.** The policy, SmolVLA, takes two camera images, the instruction text and the robot's 8 joint values, and outputs the next 50 joint commands. The red cube's row `[2, cube, 155, 103, …, hidden]` can reach it in three ways:

| Interface | What the policy receives for the red cube |
|---|---|
| A1, marker | a ring labelled "2h" drawn at pixel (155, 103) on the front image |
| A2, text | the words `[2 red cube, hidden, at <103, 155>]` appended to the instruction: row then column, the order RoboMME's own subgoal text uses ("the container at <100, 139>") |
| A3, token | the numbers (155/256, 103/256, 1) = (0.605, 0.402, 1), turned by a small trained network into one extra input vector |

A0 gives no graph, and A3feat adds the object's 768-number DINOv2 appearance vector to A3. All three carry the same four facts (id, label, position, hidden), so a difference between them is a difference of route, not of information.

**What the notebook produces.** Every stage from raw demonstrations to a trained, evaluated policy, run as a pilot on one task: A0 and A2 are trained and tested, and a printed rule says **continue** (A2's success interval lies above A0's) or **stop**. The full comparison of all five arms, four tasks and six graph-quality levels is written as `run_grid.sh` and runs only after a continue. The pilot answers only whether the graph helps at all on this task, not which interface is best.

### The approach

1. **Build the graph twice.** An oracle graph from the simulator's own object masks and depth (what perfect perception would give), and a perceived graph from this repo's perception pipeline (what a real robot would give). Compare them to measure perception error separately.
2. **Fill hidden objects** with the bound rule, checked against the simulator's truth.
3. **Make test-time quality levels.** Q0 is the oracle with the bound rule; Q1 shifts the hidden target by 20 or 50 px; Q2 moves it under a wrong container with probability 0.25 or 0.5; Q3 is the perceived graph.
4. **Train each interface on Q0, then test it at every level.** Held fixed: the model, the LoRA layers and size, the demonstrations, the training steps and the four fields per object. Varied: the interface, and at test time the graph quality.

Why test at several levels: with a correct graph, every interface may already succeed almost every time. The RoboMME paper reports that the π0.5 policy given the correct container's pixel as text succeeds on 99.2% of VideoUnmaskSwap episodes ([RoboMME](https://arxiv.org/abs/2603.04639), as tabulated in [Ledger](https://arxiv.org/abs/2609.34554)). So the comparison that can separate interfaces is how fast each one degrades as the graph gets worse.

### How the whole works

The parts, in plain words first:

- **Recorded demonstrations** (RoboMME's HDF5 files): each episode's images, depth, camera matrices and the joint commands an expert executed. 100 per task; the smoke run uses 2, the pilot all 100.
- **Simulator replay**: re-runs each recorded episode so that the simulator can also report which pixel belongs to which object (segmentation masks), which the files do not store.
- **Oracle graph builder**: turns masks and depth into the node table, frame by frame.
- **Perception pipeline**: a text-prompted detector (GroundingDINO) finds each object, a segmenter (SAM) cuts out its pixels, a tracker (Cutie) follows them frame to frame, and an image model (DINOv2) gives an appearance vector; depth turns each mask into a 3D position.
- **Bound rule** (`GraphTracker`): fills in the position of every hidden object, using only past frames, so the same code runs on recordings and live.
- **Quality levels** (`corrupt`): damage the hidden target's entry, at test time only.
- **Interface**: renders the node table as a marker image, text or tokens.
- **SmolVLA with LoRA**: the policy. LoRA adds small trainable matrices next to fixed layers; only those (and A3's small network) learn.
- **Evaluation loop**: a fresh test episode, the graph rebuilt from every frame including the video, one action chunk at a time until the episode succeeds, fails or reaches the step limit (1300, set and printed in Step 10.1).

```text
TRAINING (Steps 3-9)        [A3feat only] = used only by that arm

HDF5 demo (seed + actions)
   │
   ├────────────────────────────────┐
   ▼                                │ 50-step action chunks
simulator replay                    │ (training targets)
   │ RGB, depth, cameras, masks     │
   ▼                                │
oracle graph builder                │
   │ node table, hidden rows empty  │
   ▼                                │
bound rule                          │
   │ Q0 graph                       │
   ▼                                │
interface ◄── RGB images, joint     │
   │          state, instruction    │
   │          (+ DINOv2 vectors     │
   │           [A3feat only])       │
   │ images / text / tokens         │
   ▼                                │
SmolVLA + LoRA ◄────────────────────┘
   │ loss
   ▼
update LoRA (+ A3 network)

Side check (Step 5, not used in training):
replay RGB + depth ──► perception pipeline ──► perceived graph
                       ──► compared with the oracle graph

TESTING (Step 10)

test env ◄────────────────────────────────┐
   │ frames incl. the video, masks        │
   ▼                                      │
oracle graph builder (Q0-Q2)              │
  or perception pipeline (Q3)             │
   │ node table (+ DINOv2 [A3feat only])  │
   ▼                                      │
bound rule                                │
   ▼                                      │
quality level (damages target entry)      │
   ▼                                      │
interface                                 │
   ▼                                      │
SmolVLA (trained) ── action chunk ────────┘

episode ends: success / fail / step limit (1300, Step 10.1)
   ▼
successes over 50 episodes ──► Wilson interval ──► continue / stop
```

**Following one input through the system** (the red cube of the worked case; illustrative):

1. The replay reports that at the "cube just covered" frame the red cube has zero mask pixels in both cameras, while container 0, whose centre is within 6 cm of the cube's last position on the table, is visible. The bound rule records "red cube moves with container 0" and the offset between them.
2. Every later frame, container 0's new position plus that offset becomes the red cube's position. After the swap the row reads `[2, cube, 155, 103, …, hidden]`.
3. The interface renders that row: a ring at (155, 103) for A1, the text `[2 red cube, hidden, at <103, 155>]` (row, column) for A2, the vector (0.605, 0.402, 1) for A3.
4. SmolVLA reads the two images, the instruction (plus the object list for A2), the joint state (and one extra token per object for A3), and outputs a chunk of 50 joint commands of 8 numbers each.
5. In training, that chunk is compared with the expert's next 50 commands and only LoRA (and A3's network) is updated. In testing, the chunk is executed, the simulator returns new frames, and steps 1-4 repeat on them.

**What is shared, what decides, what changes.**

- **Shared data and who touches it:**
  - The node table `[T, N, 8]` is written by the graph builder and the bound rule, damaged by the quality level at test time, and read by the interface.
  - The replay cache is written once (Step 3) and read by every later step.
  - Checkpoints are written by training and read by evaluation.
- **What decides the next step:** during testing, the policy's action chunk; the graph is updated after every simulator step, and the episode ends on the simulator's success or failure flag or at the step limit.
- **What changes and what stays fixed:**
  - During training only LoRA and A3's network change; SmolVLA's original weights, DINOv2, the perception models and the rule stay fixed.
  - During testing nothing learns; only the graph changes, frame by frame.

### Key takeaways and why it matters

Evidence from this notebook is = ? until it runs. Lessons 2 to 4 also draw on cited work, marked where used; everything else rests on this notebook.

- **Check that a memory carries usable signal before comparing ways to deliver it.** Evidence: A0 vs A2 success in the pilot = ?. If A2 does not beat A0 on a task where memory is the whole difficulty, an interface comparison on that task would measure noise; the same check applies to any representation study.
- **When the information is right, the route may not matter; when it is wrong, the route decides.** Cited: correct coordinate text already brings π0.5 to 99.2% on VideoUnmaskSwap ([RoboMME](https://arxiv.org/abs/2603.04639)). Evidence here: degradation across Q0-Q3 per interface = ? (grid). Compare interfaces under controlled errors, not only at the best case.
- **A rule-based memory is only as good as the tracking under it, so measure that layer on its own.** Evidence: bound vs frozen at the decision frame on oracle input = ?; perceived identity accuracy through the swap = ?. A policy cannot repair a wrong binding, because the rule is not trained. The alternative is a learned memory; cited examples are [ECoMEM](https://arxiv.org/abs/2610.00801) (records written by rules, read by a trained module) and [LTM](https://arxiv.org/abs/2610.21362) (learned latent memory).
- **Compare routes with the same content and the same trainable capacity.** Here every interface carries the same four fields, and all arms share one LoRA setting; A3's extra network is reported. Without this, a gap could come from more information or more parameters rather than from the route; a comparison of planning representations that also changed content is [VLA-OS](https://arxiv.org/abs/2506.17561) (likely; reported by the independent novelty check in `_build/critique-p1.md`, not re-read for this notebook).

**Why it matters.** Explicit object memories already help VLAs on these tasks (Ledger and ECoMEM, cited above), but each feeds its memory through one route only. Knowing which route keeps working when perception makes mistakes decides how a scene graph should be wired into a policy on a real robot, where the graph will sometimes be wrong. That is the main question of the scene-graph paper this notebook prepares.

## Setup

Every function this notebook uses lives in `sg_pilot.py` next to it, so `train.py`, `eval.py` and `run_grid.sh` later run exactly the same code. Each function's source is shown once, where it is first explained. Run this cell; it only sets paths, the run mode and imports.

In [ ]:
# Setup
RUN_MODE = "smoke"  # "smoke" or "pilot"
import importlib, inspect, json, subprocess, sys, warnings

# Silence known third-party noise: toppra docstrings, torch's pynvml notice, tqdm widgets
warnings.filterwarnings("ignore", "invalid escape sequence", SyntaxWarning)
warnings.filterwarnings("ignore", "The pynvml package is deprecated", FutureWarning)
warnings.filterwarnings("ignore", "IProgress not found")
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Code, display

sys.path.insert(0, str(Path.cwd()))
import sg_pilot as sg
import pi05_policy as p5

importlib.reload(sg)
importlib.reload(p5)
CFG = sg.RUN_MODES[RUN_MODE]
TASK = CFG["task"]
# one task for smoke, one multi-task model over the four permanence tasks for the pilot (RoboMME §5.1)
TRAIN_TASKS = [TASK] if RUN_MODE == "smoke" else sg.PERMANENCE_TASKS
for d in (sg.FIG_DIR, sg.CACHE_DIR, sg.RUNS_DIR, sg.RESULTS_DIR):
    d.mkdir(parents=True, exist_ok=True)


def show(fn):
    display(Code(inspect.getsource(fn), language="python"))


print("run mode:", RUN_MODE, CFG)

## Step 1. Environment

**In:** the machine. **Out:** printed versions, one rendered simulator frame, seeds. RoboMME runs on ManiSkill, a physics simulator that renders camera images with Vulkan, so a GPU alone is not enough: the render check below must also pass.

### 1.1 Versions and GPU

In [ ]:
# Step 1.1
V = sg.versions()
for k, v in V.items():
    print(f"{k:14s} {v}")
sg.pass_fail("GPU visible", V["gpu"] is not None, V["gpu"])

### 1.2 Vulkan renders a frame, and CUDA runs a forward/backward pass

In [ ]:
# Step 1.2
r = subprocess.run(["vulkaninfo", "--summary"], capture_output=True, text=True)
print(
    "\n".join([l for l in r.stdout.splitlines() if "deviceName" in l or "apiVersion" in l][:6])
    or r.stderr[:500]
)
env = sg.make_env(TASK, "test", 0, max_steps=50)
obs, info = env.reset()
frame = sg._np(obs["front_rgb_list"][-1], np.uint8)
env.close()
plt.figure(figsize=(3, 3))
plt.imshow(frame)
plt.title(f"{TASK} test ep 0, front camera")
plt.axis("off")
plt.show()
sg.pass_fail("rendered frame not black", frame.mean() > 5, f"mean pixel {frame.mean():.1f}")
sg.pass_fail("CUDA forward/backward", np.isfinite(sg.cuda_forward_backward()), "finite loss")

### 1.3 Seeds and run mode

In [ ]:
# Step 1.3
sg.set_seed(0)
print(
    f"RUN_MODE={RUN_MODE}: task {TASK}, {CFG['train_episodes']} training episodes, "
    f"{CFG['steps'] or 'measured in 9.6'} training steps per arm, {CFG['eval_episodes']} evaluation episodes"
)
sg.pass_fail("seeds set", True, 0)

## Step 2. Look at the data

**In:** the RoboMME demonstrations. **Out:** what one VideoUnmaskSwap episode contains and the frame where memory is first needed.

The task itself (the video, the swap, the instruction) is described in At a Glance; this step shows it in the data.

**Running example.** Every step below follows the same episode, the first training episode `EP`, and its target cube, so each new output can be compared with the previous one.

Two copies of the same 1,600 demonstrations exist: the raw HDF5 files (`Yinpei/robomme_data_h5`) and a LeRobot-format copy (`lerobot/robomme`). This notebook reads the HDF5 files, because they also hold depth and camera parameters, which the scene graph needs; the LeRobot copy supplies the normalisation statistics SmolVLA expects (Step 8).

### 2.1 Episodes per task and split

In [ ]:
# Step 2.1
from robomme.env_record_wrapper import BenchmarkEnvBuilder

rows = []
for task in sg.PERMANENCE_TASKS:
    rows.append(
        [task]
        + [
            BenchmarkEnvBuilder(env_id=task, dataset=s).get_episode_num()
            for s in ("train", "val", "test")
        ]
    )
print(f"{'task':18s} train  val  test")
for r in rows:
    print(f"{r[0]:18s} {r[1]:5d} {r[2]:4d} {r[3]:5d}")
n_test = dict((r[0], r[3]) for r in rows)[TASK]
sg.pass_fail("test split has >= 50 episodes for the pilot", n_test >= 50, n_test)

### 2.2 One training episode: every stored field

In [ ]:
# Step 2.2
import h5py

h5_path = sg.download_h5(TASK, sg.CACHE_DIR / "h5")
h5 = h5py.File(h5_path, "r")
EPS = sg.h5_episode_indices(h5)
EP = EPS[0]
h5ep = sg.read_h5_episode(h5, EP)
print(
    h5_path.name,
    "episodes:",
    len(EPS),
    "| using episode",
    EP,
    "| timesteps:",
    len(h5ep["front_rgb"]),
)
meaning = {
    "front_rgb": "front camera RGB",
    "wrist_rgb": "wrist camera RGB",
    "front_depth": "front depth (mm)",
    "joint_state": "7 joint angles",
    "gripper_state": "2 finger widths (m)",
    "is_gripper_close": "gripper closed",
    "eef_state": "end effector xyz + rpy",
    "front_camera_extrinsic": "world->camera 3x4",
    "joint_action": "action: 7 joints + gripper",
    "is_video_demo": "frame of the conditioning video",
}
for k, m in meaning.items():
    x = next((v for v in h5ep[k] if v is not None), None)
    print(f"{k:24s} {str(np.shape(x)):14s} {str(np.asarray(x).dtype):8s} {m}")
print("front intrinsic\n", h5ep["front_camera_intrinsic"])
from lerobot.datasets import LeRobotDatasetMetadata

meta = LeRobotDatasetMetadata(sg.LEROBOT_REPO)
print("lerobot/robomme fps:", meta.fps, "| features:", list(meta.features))

### 2.3 Action, state and instruction

In [ ]:
# Step 2.3
A = sg.policy_actions(h5ep)
print(
    "actions:", A.shape, "= [policy steps, 7 joint angles (rad) + gripper (-1 close, +1 open)]"
)
print("state used by the policy: 7 joint angles + first finger width = 8 values")
print("instruction:", h5ep["task_goal"])
sg.pass_fail("action dimension 8", A.shape[1] == 8, A.shape)

### 2.4 Actions over time

In [ ]:
# Step 2.4
fig, axes = plt.subplots(2, 4, figsize=(12, 4), sharex=True)
for i, ax in enumerate(axes.flat):
    ax.plot(A[:, i])
    ax.set_title(f"joint {i+1}" if i < 7 else "gripper (-1 closed)", fontsize=8)
for ax in axes[1]:
    ax.set_xlabel("policy step")
fig.suptitle(f"{TASK} ep {EP}: commanded joint angles (rad) and gripper")
fig.tight_layout()
fig.savefig(sg.FIG_DIR / "02_actions.png", dpi=120)
plt.show()
sg.pass_fail("actions finite", np.isfinite(A).all(), A.shape)

### 2.5 Six moments of the episode

In [ ]:
# Step 2.5
vid = np.array(h5ep["is_video_demo"])
nv = int(vid.sum())
moments = {
    "start": 0,
    "objects visible": nv // 8,
    "covers lowered": nv // 3,
    "mid swap": nv // 2 + nv // 8,
    "swap done": max(nv - 2, 0),
    "end": len(vid) - 1,
}
fig, axes = plt.subplots(2, 6, figsize=(15, 5.4))
for j, (name, t) in enumerate(moments.items()):
    axes[0, j].imshow(h5ep["front_rgb"][t])
    axes[0, j].set_title(f"{name}\nframe {t}", fontsize=8)
    axes[1, j].imshow(h5ep["wrist_rgb"][t])
for ax in axes.flat:
    ax.set_xticks([])
    ax.set_yticks([])
axes[0, 0].set_ylabel("front")
axes[1, 0].set_ylabel("wrist")
fig.suptitle(
    f"{TASK} ep {EP}: frames 0..{nv-1} are the conditioning video, later frames are the robot acting"
)
fig.tight_layout()
fig.savefig(sg.FIG_DIR / "02_frames.png", dpi=120)
plt.show()
print("video frames:", nv, "| acting frames:", len(vid) - nv)
sg.pass_fail("episode has a conditioning video", nv > 0, nv)

Moments are picked by position in the video, not by detected events; read the titles against the images.

### 2.6 Subgoals over time

In [ ]:
# Step 2.6
for a, b, s in sg.subgoal_ranges(h5ep["grounded_subgoal"]):
    print(f"frames {a:4d}-{b:4d}  {s}")
sg.pass_fail(
    "subgoals present",
    any(h5ep["grounded_subgoal"]),
    len(sg.subgoal_ranges(h5ep["grounded_subgoal"])),
)

### 2.7 What must be remembered, and when

In [ ]:
# Step 2.7
t_star_h5 = int(np.argmax(~vid))
print("first acting frame (decision frame t*, HDF5 indexing):", t_star_h5)
sg.pass_fail("t* after the video", t_star_h5 == nv, t_star_h5)

**How to read the output above.** = ? (one sentence after the run: which container hides the target at the end of the video in this episode, and that the policy first needs it at frame t*, when the image shows only identical containers.)

**Check yourself.** Looking only at the "end" frame of the grid in 2.5, can you tell which container hides the target? If not, that is exactly the gap the scene graph fills.

## Step 3. Replay in the simulator to get privileged data

**In:** the episode's seed and recorded actions. **Out:** every frame again, now with segmentation masks, depth in metres, camera parameters and the gripper state.

A **segmentation map** is an image of the same size as the camera image in which each pixel holds the id of the object it shows instead of a colour. For example, a 3 x 3 corner around a red cube might read:

```text
camera pixels            segmentation ids
grey  grey  red          0   0  17
grey  red   red          0  17  17
grey  grey  grey         0   0   0
```

Here 17 would be the red cube's id (the real ids of this episode are printed in 3.1). Counting an object's pixels says whether it is visible; their positions say where. The HDF5 file already has depth and cameras; segmentation exists only inside the simulator, so the episode is replayed: `reset()` plays the conditioning video, then the recorded actions are stepped.

### 3.1 The environment's observation fields

In [ ]:
# Step 3.1
show(sg.make_env)
env = sg.make_env(TASK, "train", EP)
obs, info = env.reset()
print("obs keys:", list(obs))
print("info keys:", list(info))
ms = obs["maniskill_obs"][-1]
print(
    "maniskill sensor_data cameras:",
    list(ms["sensor_data"]),
    "| fields:",
    list(ms["sensor_data"]["base_camera"]),
)
scene = sg.scene_objects(env)
env.close()
print("objects:", [(o["node"], o["text"], o["name"]) for o in scene["objects"]])
sg.pass_fail(
    "segmentation available", "segmentation" in ms["sensor_data"]["base_camera"], "base_camera"
)

### 3.2 Replay the episode

In [ ]:
# Step 3.2
show(sg.replay_episode)
rep = sg.replay_episode(TASK, EP, A, split="train")
for k in [
    "front_rgb",
    "front_depth_m",
    "front_seg",
    "E_front",
    "joint_state",
    "gripper_state",
    "ee_xyz",
    "is_video",
    "policy_frame",
    "actions",
]:
    print(f"{k:14s} {str(getattr(rep, k).shape):20s}")
print("K_front\n", rep.K_front, "\nstatus:", rep.status, "| goal:", rep.task_goal)
print(f"replayed {len(rep.policy_frame)} of {len(A)} recorded actions; the env stops at success")
sg.pass_fail(
    "replay ends in success", rep.status == "success", (rep.status, len(rep.policy_frame), len(A))
)

### 3.3 The replay matches the recorded episode

**Why compare.** The scene graph is built from the replay, but the policy is trained on recorded actions. If the replay drifted from the recording, graphs and actions would describe different episodes. The check uses frames before the covers lower (where a wrong cube placement would be visible), and checks the target cube's colour at its frame-0 position.

In [ ]:
# Step 3.3
n_h5, n_rep = len(h5ep["front_rgb"]), len(rep.front_rgb)
idx = [
    0,
    nv // 8,
    nv // 2,
    nv,
    min(n_h5, n_rep) - 1,
]  # the first two are before the covers lower
diffs = [
    float(
        np.abs(
            rep.front_rgb[t].astype(int) - np.asarray(h5ep["front_rgb"][t]).astype(int)
        ).mean()
    )
    for t in idx
]
print("frames:", idx, "| mean |RGB diff|:", np.round(diffs, 2))
og0, _ = sg.oracle_graph(rep)
targets = sg.target_cube_nodes(rep.scene, rep.task_goal)  # ordered; see 4.1
tgt = targets[0]
u, v = og0[0, tgt, sg.F_U], og0[0, tgt, sg.F_V]
pix = np.asarray(h5ep["front_rgb"][0])[int(round(v)), int(round(u))]
colour = rep.scene["objects"][tgt]["color"]
fig, ax = plt.subplots(1, 2, figsize=(6, 3))
ax[0].imshow(h5ep["front_rgb"][nv // 2])
ax[0].set_title("recorded (HDF5)", fontsize=8)
ax[1].imshow(rep.front_rgb[nv // 2])
ax[1].set_title("replayed", fontsize=8)
for a in ax:
    a.axis("off")
fig.suptitle(f"mid-swap frame {nv // 2}")
plt.show()
sg.pass_fail("replay is a prefix of the recording", n_rep <= n_h5, (n_h5, n_rep))
sg.pass_fail("mean |RGB diff| < 5/255", max(diffs) < 5, round(max(diffs), 2))
sg.pass_fail(
    f"{colour} cube pixel at its frame-0 position",
    int(np.argmax(pix)) == ["red", "green", "blue"].index(colour),
    pix.tolist(),
)

**How to read the output above.** = ? (after the run: how many recorded frames follow the replay's success, the largest pixel difference, and the cube-colour check.)

### 3.4 Depth

In [ ]:
# Step 3.4
# Colour scale clipped to the 2nd-90th depth percentile: the far wall (~2.9 m) would flatten the table
d = rep.front_depth_m[rep.front_depth_m > 0]
lo, hi = np.percentile(d, [2, 90])
fig, axes = plt.subplots(2, 3, figsize=(9, 6))
for j, t in enumerate([0, nv // 2, n_rep - 1]):
    axes[0, j].imshow(rep.front_rgb[t])
    axes[0, j].set_title(f"frame {t}", fontsize=8)
    im = axes[1, j].imshow(rep.front_depth_m[t], cmap="viridis", vmin=lo, vmax=hi)
    plt.colorbar(im, ax=axes[1, j], fraction=0.046, label="m")
for a in axes.flat:
    a.axis("off")
fig.suptitle(f"front camera RGB and depth (metres, colour clipped to {lo:.2f}-{hi:.2f} m)")
fig.tight_layout()
plt.show()
print(f"depth range {d.min():.3f} to {d.max():.3f} m (converted from int16 mm)")
sg.pass_fail(
    "depth in metres and plausible", 0.05 < np.median(d) < 3.0, round(float(np.median(d)), 3)
)

### 3.5 Segmentation

In [ ]:
# Step 3.5
from matplotlib.patches import Patch

cmap = plt.get_cmap("tab10")
# Cubes in their own colour; containers in tab10 grey, olive, cyan so no hue repeats
CUBE_RGB = {"red": (0.85, 0.1, 0.1), "green": (0.1, 0.75, 0.1), "blue": (0.15, 0.3, 0.95)}


def obj_colour(o):
    return CUBE_RGB.get(o.get("color")) or cmap(7 + o["node"] % 3)[:3]


fig, axes = plt.subplots(1, 3, figsize=(10, 3.6))
for j, t in enumerate([0, nv // 2, n_rep - 1]):
    over = rep.front_rgb[t].copy() // 3
    for o in rep.scene["objects"]:
        m = np.isin(rep.front_seg[t], rep.scene["seg_ids"][o["node"]])
        over[m] = (np.array(obj_colour(o)) * 255).astype(np.uint8)
    axes[j].imshow(over)
    axes[j].set_title(f"frame {t}", fontsize=8)
    axes[j].axis("off")
fig.legend(
    handles=[
        Patch(color=obj_colour(o), label=f"{o['node']} {o['text']}")
        for o in rep.scene["objects"]
    ],
    loc="lower center",
    ncol=len(rep.scene["objects"]),
    fontsize=7,
)
fig.suptitle("segmentation masks of the task objects (others dimmed)")
plt.show()
sg.pass_fail(
    "every object has segmentation ids",
    all(rep.scene["seg_ids"][o["node"]] for o in rep.scene["objects"]),
    rep.scene["seg_ids"],
)

### 3.6 Missing fields and the replay cache

In [ ]:
# Step 3.6
missing = [
    k
    for k in ["front_seg", "wrist_seg", "front_depth_m", "E_front"]
    if getattr(rep, k).size == 0
]
print("missing fields:", missing or "none")
for ep in EPS[: CFG["train_episodes"]]:
    path = sg.CACHE_DIR / TASK / f"ep{ep:04d}_replay.npz"
    if not path.exists():
        e_h5 = sg.read_h5_episode(h5, ep)
        sg.save_replay(sg.replay_episode(TASK, ep, sg.policy_actions(e_h5)), path)
for ep in EPS[: CFG["train_episodes"]]:
    r = sg.load_replay(sg.CACHE_DIR / TASK / f"ep{ep:04d}_replay.npz")
    print(f"ep {ep}: {len(r.front_rgb)} frames, {len(r.actions)} actions, status {r.status}")
sg.pass_fail("no missing field", not missing, missing)

## Step 4. Oracle scene graph

**In:** the replay. **Out:** `oracle_graph` with shape `[T, N, 8]`: for each of T frames and N objects, the fields `[id, label_id, u, v, x, y, z, hidden]`. Here (u, v) is the pixel in the front image (u = column, v = row) and (x, y, z) the world position in metres (x forward, y left, z up, table top at z = 0). The positions come from the simulator's masks and depth, so they are the best a perfect perception system could produce.

### 4.1 Objects and node ids

Some episodes ask for two cubes in turn ("... hiding the blue cube, finally pick up another container hiding the red cube"). The graph always holds every cube; wherever a single target is needed (the rule errors in Step 6, the corruptions in Step 7, the metrics in Step 10), it is the **active target**: the first cube named, until its container is picked up (pose z above 0.15 m, the test RoboMME itself uses), then the next.

In [ ]:
# Step 4.1
for o in rep.scene["objects"]:
    print(
        f"node {o['node']}: {o['text']:12s} label_id {o['label_id']}  simulator name {o['name']}"
    )
print("cube -> container it starts under:", rep.scene["cube_to_bin"])
print("targets in pick order:", [(n, rep.scene["objects"][n]["text"]) for n in targets])
show(sg.ActiveTarget)
active = sg.active_targets(rep, targets)
switches = [(t, int(active[t])) for t in range(1, len(active)) if active[t] != active[t - 1]]
print("active target switches (frame, new target):", switches or "none (one target)")
import re

goals = sg.h5_task_goals(h5)
n_two = sum(len(re.findall(r"\b(?:red|green|blue) cube", g)) == 2 for g in goals.values())
print(f"two-target episodes in the training split: {n_two} of {len(goals)}")
sg.pass_fail("first target active at t*", active[t_star_h5] == targets[0], int(active[t_star_h5]))

### 4.2 Projecting world points into the image

Every position check in this notebook moves between 3D positions and pixels, in four rungs:

1. **In words.** A camera maps each ray of light to one pixel; the further a point is from the camera, the closer to the image centre it lands.
2. **In camera coordinates** (z straight ahead, x to the right, y down), a point (x, y, z) lands at

   $u = c_x + f_x \, x / z, \qquad v = c_y + f_y \, y / z$

   where $f_x, f_y$ (focal lengths in pixels) and $c_x, c_y$ (the image centre) are the entries of the intrinsic matrix $K = \begin{bmatrix} f_x & 0 & c_x \\ 0 & f_y & c_y \\ 0 & 0 & 1 \end{bmatrix}$. With the $K$ printed in 3.2, a point 0.5 m ahead and 0.1 m to the right lands $f_x \cdot 0.1 / 0.5$ pixels right of the centre.
3. **From the world.** Objects are stored in world coordinates, so they are first moved into camera coordinates with the extrinsic matrix $E = [R \mid t]$ (3 x 4): $p_{cam} = R \, p_{world} + t$. RoboMME gives $E$ in this world-to-camera OpenCV form (`extrinsic_cv`).
4. **Backwards.** With a pixel's depth $z$, the same equations solve for $x, y$, and $p_{world} = R^\top (p_{cam} - t)$; `cam_to_world` builds that inverse as a 4 x 4 matrix.

`project_points` is rung 3 then rung 2; `backproject_mask` is rung 4, averaged over an object's pixels. The figure checks both: each object's 3D position, projected back, should land on its own mask.

In [ ]:
# Step 4.2
show(sg.project_points)
show(sg.backproject_mask)
og, full_xyz = sg.oracle_graph(rep)
errs = []
for t in [0, nv // 2, n_rep - 1]:
    for n in range(og.shape[1]):
        if og[t, n, sg.F_HIDDEN] == 0 and np.all(np.isfinite(og[t, n, sg.F_X : sg.F_Z + 1])):
            uv = sg.project_points(og[t, n, sg.F_X : sg.F_Z + 1], rep.K_front, rep.E_front[t])[
                0
            ]
            errs.append(np.linalg.norm(uv - og[t, n, sg.F_U : sg.F_V + 1]))
fig, axes = plt.subplots(1, 3, figsize=(9, 3.2))
for a, t in zip(axes, [0, nv // 2, n_rep - 1]):
    a.imshow(rep.front_rgb[t])
    for n in range(og.shape[1]):
        if og[t, n, sg.F_HIDDEN] == 0:
            uv = sg.project_points(og[t, n, sg.F_X : sg.F_Z + 1], rep.K_front, rep.E_front[t])[
                0
            ]
            a.plot(*uv, "x", color="white")
            a.text(uv[0] + 3, uv[1], str(n), color="white", fontsize=7)
    a.set_title(f"frame {t}", fontsize=8)
    a.axis("off")
fig.suptitle("projected 3D positions (x) on the front image")
plt.show()
sg.pass_fail(
    "projection lands on the mask centroid (median px)",
    np.median(errs) < 3,
    round(float(np.median(errs)), 2),
)

### 4.3 When each object is hidden

In [ ]:
# Step 4.3
H = og[..., sg.F_HIDDEN].T
plt.figure(figsize=(10, 2))
plt.imshow(H, aspect="auto", cmap="Greys", interpolation="nearest")
plt.yticks(
    range(len(H)), [f"{o['node']} {o['text']}" for o in rep.scene["objects"]], fontsize=7
)
plt.axvline(nv, color="r", lw=1)
plt.xlabel("frame (red line: robot starts acting)")
plt.title("hidden (dark) = no mask pixels in either camera")
plt.tight_layout()
plt.show()
sg.pass_fail(
    "target hidden at t*",
    og[t_star_h5, tgt, sg.F_HIDDEN] == 1,
    int(og[t_star_h5, tgt, sg.F_HIDDEN]),
)

### 4.4 The graph's layout and values

In [ ]:
# Step 4.4
print("oracle_graph", og.shape, "fields", sg.NODE_FIELDS)
np.set_printoptions(precision=3, suppress=True, linewidth=140)
for t in [0, nv // 3, nv // 2, t_star_h5]:
    print(f"\nframe {t}")
    print(og[t])

### 4.5 The hidden cube moves with its container

In [ ]:
# Step 4.5
bin_ = rep.scene["cube_to_bin"][tgt]
true_uv = np.array(
    [sg.project_points(full_xyz[t, tgt], rep.K_front, rep.E_front[t])[0] for t in range(n_rep)]
)
fig, ax = plt.subplots(figsize=(9, 3))
ax.plot(true_uv[:, 0], label=f"target {rep.scene['objects'][tgt]['text']} u (full oracle)")
for o in rep.scene["objects"]:
    if o["label_id"] == 0:
        ax.plot(og[:, o["node"], sg.F_U], "--", lw=1, label=f"container {o['node']} u")
ax.fill_between(
    range(n_rep),
    *ax.get_ylim(),
    where=og[:, tgt, sg.F_HIDDEN] == 1,
    color="grey",
    alpha=0.15,
    label="target hidden",
)
ax.set_xlabel("frame")
ax.set_ylabel("u (px)")
ax.legend(fontsize=7)
ax.set_title("target position while hidden")
fig.tight_layout()
fig.savefig(sg.FIG_DIR / "04_oracle_track.png", dpi=120)
plt.show()
moved = np.nanmax(true_uv[:, 0]) - np.nanmin(true_uv[:, 0])
sg.pass_fail(
    "hidden target moves during the swap (u range px)", moved > 10, round(float(moved), 1)
)

The full oracle gives the hidden cube its container's position, because the simulator carries each cube with its container through the swap (`VideoUnmaskSwap.step`, `lift_and_drop_objectA_onto_objectB`).

## Step 5. Perceived scene graph with the existing pipeline

**In:** the replayed front images, depth and cameras. **Out:** `perceived_graph`, in the same `[T, N, 8]` layout, built by this repo's HODOR pipeline instead of the simulator: what a real robot would have. Each stage, by what goes in and what comes out:

| Stage | In | Out | When it runs |
|---|---|---|---|
| GroundingDINO | image [512, 512, 3] + a phrase, e.g. `"container."` | boxes [k, 4] with scores | once per class, at the first frame the class is found |
| SAM | image + one box | mask [512, 512] (true/false per pixel) | once per accepted box |
| Cutie | each new frame | that slot's mask in the new frame | every frame, one tracker per slot, started from SAM's mask |
| DINOv2 | image resized to 224 x 224 | global vector [768] + patch vectors [256, 768] (a 16 x 16 grid) | every frame |
| depth | a slot's mask + depth + K, E | 3D centroid [3] | every frame |

A **slot** is one entry of the class list: one tracked object. Listing `"container."` once per container gives each container its own slot.

**Why identical containers are hard.** Cutie follows each mask by appearance. When two identical containers cross during the swap, appearance cannot tell them apart, so a slot may jump to the other container. Step 5.9 measures how often this happens instead of assuming it does not.

**What had to change.** The repo was written for DROID's rotated wrist camera: `_depth_to_pointcloud` subtracts 150 from $c_x$ (DROID's crop) and `_add_centroids` resizes masks to 800 x 800 and rotates them 180 degrees. `RoboMMEProcessor` overrides only these two methods (marked `# changed`); no repo file is edited.

### 5.1 Camera and class configuration

In [ ]:
# Step 5.1
show(sg.make_robomme_perception_config)
colors = [o["color"] for o in rep.scene["objects"] if o["label_id"] == 1]
pcfg = sg.make_robomme_perception_config(rep.scene["n_bins"], colors, scale=2)
print("camera: front (fixed, sees every container; the wrist camera moves with the arm)")
CLASSES = pcfg["task"]["classes"]
print("classes (slot order):", CLASSES, "| source: manual config")
print("container minimum box area (px^2 at 512):", pcfg["perception"]["class_area_ranges"])
print(
    "detector image: 512 x 512 (front 256 px upscaled x2); global area range",
    pcfg["perception"]["global_area_min"],
    pcfg["perception"]["global_area_max"],
)

### 5.2 One frame as the pipeline's arguments

The pipeline runs in a separate, long-lived process (`perception_worker.py`, in its own Python environment): GroundingDINO needs an older `transformers` than SmolVLA, so the two cannot share one. The worker loads its four models once; `make_robomme_processor` returns a handle whose `process_frame` has the same arguments as the repo's `HODORProcessor.process_frame`.

In [ ]:
# Step 5.2
show(sg.robomme_processor_class)
show(sg.make_robomme_processor)
proc = sg.make_robomme_processor(pcfg)
print("perception worker:", sg.get_perception_worker().ready)
args = dict(
    image=rep.front_rgb[0],
    depth_m=rep.front_depth_m[0],
    K=rep.K_front,
    T_cam2world=sg.cam_to_world(rep.E_front[0]),
)
for k, a in args.items():
    print(
        f"{k:12s} shape {np.shape(a)} dtype {np.asarray(a).dtype} range [{np.min(a):.3f}, {np.max(a):.3f}]"
    )

### 5.3 to 5.8 Running the pipeline over the episode

In [ ]:
# Step 5.3
show(sg.perceived_graph)
moments_rep = [0, nv // 8, nv // 3, nv // 2 + nv // 8, max(nv - 2, 0), n_rep - 1]
pg, pfeat, seeds, label_masks = sg.perceived_graph(
    rep, proc, scale=2, keep_masks=set(moments_rep)
)
import cv2

img0 = cv2.resize(rep.front_rgb[0], (512, 512), interpolation=cv2.INTER_CUBIC)
print("seed events (class, frame, score):", [(e["class"], e["frame"], e["gdino_conf"]) for e in seeds])
# Cubes are found at frame 0; containers only enter the view later, so draw each seed frame
seed_frames = sorted({e["frame"] for e in seeds})
fig, axes = plt.subplots(1, len(seed_frames), figsize=(4 * len(seed_frames), 4), squeeze=False)
for a, f in zip(axes[0], seed_frames):
    a.imshow(cv2.resize(rep.front_rgb[f], (512, 512), interpolation=cv2.INTER_CUBIC))
    for e in [e for e in seeds if e["frame"] == f]:
        x0, y0, x1, y1 = e["box_xyxy"]
        a.add_patch(plt.Rectangle((x0, y0), x1 - x0, y1 - y0, fill=False, color="yellow"))
        a.text(x0, y0 - 3, f"{e['class']} {e['gdino_conf']:.2f}", color="yellow", fontsize=6)
    a.set_title(f"frame {f}: GroundingDINO boxes kept (class, score)", fontsize=8)
    a.axis("off")
fig.tight_layout()
fig.savefig(sg.FIG_DIR / "05_detect.png", dpi=120)
plt.show()
show(sg.assign_slots_seeded)
assign_seed, seed_t = sg.assign_slots_seeded(pg, og)
for s_ in range(len(CLASSES)):
    n_ = assign_seed.get(s_)
    print(
        f"slot {s_ + 1} {CLASSES[s_]:12s} seeded at frame {seed_t.get(s_)} -> oracle node {n_}"
        + (f" ({rep.scene['objects'][n_]['text']})" if n_ is not None else "")
    )
cont_nodes = sorted(assign_seed[s] for s in assign_seed if CLASSES[s].startswith("container"))
sg.pass_fail(
    "container slots seeded on distinct containers",
    len(cont_nodes) == len(set(cont_nodes)) == rep.scene["n_bins"],
    cont_nodes,
)
cube_ok = [
    rep.scene["objects"][assign_seed[s]]["text"] == CLASSES[s].rstrip(".")
    for s in range(len(CLASSES))
    if not CLASSES[s].startswith("container") and s in assign_seed
]
sg.pass_fail(
    "each cube slot seeded on the cube of its colour",
    len(cube_ok) == len(colors) and all(cube_ok),
    cube_ok,
)


In [ ]:
# Step 5.4
lm = label_masks[0]
plt.figure(figsize=(4, 4))
plt.imshow(img0)
plt.imshow(
    np.ma.masked_where(lm == len(CLASSES), lm),
    cmap="tab10",
    alpha=0.5,
    vmin=0,
    vmax=9,
)
for s in range(len(CLASSES)):
    ys, xs = np.nonzero(lm == s)
    if len(xs):
        plt.text(xs.mean(), ys.mean(), str(s + 1), color="white", fontsize=8)
plt.title("frame 0: SAM masks with slot numbers")
plt.axis("off")
plt.show()

In [ ]:
# Step 5.5
fig, axes = plt.subplots(1, 6, figsize=(15, 3))
for a, t in zip(axes, moments_rep):
    a.imshow(cv2.resize(rep.front_rgb[t], (512, 512)))
    m = label_masks.get(t)
    if m is not None:
        a.imshow(
            np.ma.masked_where(m == len(CLASSES), m),
            cmap="tab10",
            alpha=0.5,
            vmin=0,
            vmax=9,
        )
    a.set_title(f"frame {t}", fontsize=8)
    a.axis("off")
fig.suptitle("Cutie tracking: each colour is one slot")
fig.tight_layout()
plt.show()
lost = {s: np.nonzero(pg[:, s, sg.F_HIDDEN] == 1)[0] for s in range(pg.shape[1])}
for s, fr in lost.items():
    print(
        f"slot {s+1} ({CLASSES[s]}): no mask in {len(fr)} frames"
        + (f", first {fr[:5].tolist()}" if len(fr) else "")
    )

In [ ]:
# Step 5.6
t = nv // 2
fig, ax = plt.subplots(1, 2, figsize=(8, 3.6))
for n in range(og.shape[1]):
    if og[t, n, sg.F_HIDDEN] == 0:
        ax[0].scatter(og[t, n, sg.F_X], og[t, n, sg.F_Y], marker="o", label=f"oracle {n}")
for s in range(pg.shape[1]):
    if pg[t, s, sg.F_HIDDEN] == 0:
        ax[0].scatter(pg[t, s, sg.F_X], pg[t, s, sg.F_Y], marker="x", label=f"slot {s+1}")
ax[0].set_xlabel("x (m, forward)")
ax[0].set_ylabel("y (m, left)")
ax[0].set_title(f"top-down, frame {t}", fontsize=8)
ax[0].legend(fontsize=5)
ax[1].imshow(rep.front_rgb[t])
for s in range(pg.shape[1]):
    if pg[t, s, sg.F_HIDDEN] == 0:
        ax[1].plot(pg[t, s, sg.F_U], pg[t, s, sg.F_V], "x", color="yellow")
ax[1].set_title("perceived centroids projected back", fontsize=8)
ax[1].axis("off")
fig.tight_layout()
plt.show()

In [ ]:
# Step 5.7
import torch, torch.nn.functional as Fnn

s = next(s_ for s_ in range(len(CLASSES)) if (label_masks[0] == s_).any())  # visible at frame 0
mask = torch.from_numpy((label_masks[0] == s).astype(np.float32))[None, None]
grid = Fnn.adaptive_max_pool2d(mask, (16, 16))[0, 0].numpy() > 0.5
fig, ax = plt.subplots(1, 2, figsize=(7, 3.4))
ax[0].imshow(cv2.resize(rep.front_rgb[0], (224, 224)))
ax[0].set_title("224 x 224 input to DINOv2", fontsize=8)
ax[1].imshow(grid, cmap="Greys")
ax[1].set_title(f"slot {s+1} ({CLASSES[s]}): patches averaged ({grid.sum()} of 256)", fontsize=8)
for a in ax:
    a.axis("off")
plt.show()
vec = pfeat[0, s]
print(
    f"feature vector: shape {vec.shape}, norm {np.linalg.norm(vec):.2f}, first 8 values {np.round(vec[:8], 3)}"
)

In [ ]:
# Step 5.8
pg_n, pfeat_n = sg.reorder_to_oracle(pg, pfeat, assign_seed, og.shape[1])
for t in [0, nv // 3, nv // 2, t_star_h5]:
    print(f"\nframe {t}   oracle (u, v, hidden)  |  perceived (u, v, hidden)")
    for n in range(og.shape[1]):
        print(
            f"  node {n}: {og[t, n, [2, 3, 7]].round(1)}  |  {pg_n[t, n, [2, 3, 7]].round(1)}"
        )

### 5.9 How good is the perceived graph?

**How to read the output above.** = ? (after the run: identity accuracy over the post-swap frames, how many frames had two slots on one object, and the median position error. The error compares the centroid of the visible surface with the oracle position, so even perfect tracking leaves a small offset.)

In [ ]:
# Step 5.9
post = list(range(nv // 2, n_rep))
ida = sg.identity_accuracy(pg, og, assign_seed, post)
print(ida)
# per-frame centroid error of matched, visible slots (visible-surface centroid vs oracle position)
err_tn = np.full((n_rep, og.shape[1]), np.nan)
for t in range(n_rep):
    for n in range(og.shape[1]):
        if pg_n[t, n, sg.F_HIDDEN] == 0 and og[t, n, sg.F_HIDDEN] == 0:
            err_tn[t, n] = np.linalg.norm(
                pg_n[t, n, sg.F_X : sg.F_Z + 1] - og[t, n, sg.F_X : sg.F_Z + 1]
            )
n_nan_xyz = int(np.sum((pg_n[..., sg.F_HIDDEN] == 0) & ~np.isfinite(pg_n[..., sg.F_X])))
fig, ax = plt.subplots(figsize=(9, 3))
for n in range(og.shape[1]):
    ax.plot(err_tn[:, n] * 100, lw=1, label=f"node {n} {rep.scene['objects'][n]['text']}")
ax.axvline(t_star_h5, color="r", lw=1, label="t*")
ax.set_xlabel("frame")
ax.set_ylabel("centroid error (cm)")
ax.set_title("perceived vs oracle position per frame (gaps: slot or object hidden)")
ax.legend(fontsize=6, ncol=2)
fig.tight_layout()
plt.show()
print(
    f"median position error {np.nanmedian(err_tn):.3f} m over {int(np.isfinite(err_tn).sum())} "
    f"object-frames; slot-frames with a mask but no valid depth point: {n_nan_xyz}"
)
lost_frames = sorted({int(t) for s_ in assign_seed for t in np.nonzero(pg[:, s_, sg.F_HIDDEN] == 1)[0] if t >= seed_t[s_]})
print("frames where a seeded slot had no mask:", len(lost_frames), lost_frames[:20], "...")
sg.pass_fail(
    "identity accuracy measured",
    np.isfinite(ida["identity_accuracy"]),
    ida["identity_accuracy"],
)

## Step 6. Hidden-object update rules

**In:** a graph whose hidden objects have no position. **Out:** the same graph with a position for every hidden object, from one of two rules:
- **frozen**, the seed paper's rule: keep the last seen position;
- **bound**: when an object disappears, attach it to the object it disappeared under (or to the gripper, if the gripper closed on it) and move it with that object.

Both rules run frame by frame and only look at past frames, so the same code also runs online during evaluation (Step 10).

**Walkthrough with the running example** (frame numbers are printed by 6.3; = ? until the run):

1. Before the covers drop, the target cube is visible; every frame its position is stored as `last`.
2. At frame a (= ?), its mask disappears. At that same frame a container C (= ?) is visible, and C's position on the table (x, y) lies within `r_bind` = 6 cm of the cube's last (x, y). **bound** records partner = C and offset = cube minus C. **frozen** records nothing.
3. During the swap, C moves. Each frame, **bound** sets cube = C's current position + offset; **frozen** keeps the old `last`.
4. At t*, **bound** reports C's position; **frozen** reports where the cube was before the swap, which is now under a different container.

In `GraphTracker.update` below, step 2 is the loop over `hidden & ~self.prev_hidden` (objects that just disappeared), and step 3 is `self.last[n] = self.last[p] + self.offset[n]`.

**Limit.** If `r_bind` were smaller than the gap between the cube's last position and its container's centre, no partner would be found and bound would behave exactly like frozen.

**Check yourself.** Before running 6.5, predict where each rule's marker will be in the t* image.

In [ ]:
# Step 6.1
show(sg.GraphTracker)

In [ ]:
# Step 6.2
gated = og.copy()  # visibility-gated oracle: oracle values while visible, NaN while hidden
gated[gated[..., sg.F_HIDDEN] == 1, sg.F_U : sg.F_Z + 1] = np.nan
grip_w = rep.gripper_state.sum(1)
frozen_g, ev_f = sg.apply_rule(gated, "frozen", grip_w, rep.ee_xyz, rep.K_front, rep.E_front)
bound_g, ev_b = sg.apply_rule(gated, "bound", grip_w, rep.ee_xyz, rep.K_front, rep.E_front)

In [ ]:
# Step 6.3
for t, n, p in ev_b:
    print(f"frame {t}: node {n} ({rep.scene['objects'][n]['text']}) hidden -> bound to {p}")
true_bin = rep.scene["cube_to_bin"][tgt]
ev_t = [p for _, n, p in ev_b if n == tgt]
sg.pass_fail(
    "binding names the true container",
    bool(ev_t) and ev_t[0] == true_bin,
    (ev_t[:1], true_bin),
)

In [ ]:
# Step 6.4
pxf, mf = sg.target_error(frozen_g, full_xyz, active, rep.K_front, rep.E_front)
pxb, mb = sg.target_error(bound_g, full_xyz, active, rep.K_front, rep.E_front)
fig, ax = plt.subplots(figsize=(9, 3))
ax.plot(pxf, label="frozen")
ax.plot(pxb, label="bound")
ax.axvline(t_star_h5, color="r", lw=1, label="t* (first action)")
ax.set_xlabel("frame")
ax.set_ylabel("active target error (px)")
ax.set_title("hidden target position error vs full oracle")
ax.legend(fontsize=7)
fig.tight_layout()
fig.savefig(sg.FIG_DIR / "06_rules.png", dpi=120)
plt.show()

In [ ]:
# Step 6.5
t = t_star_h5
tru = sg.project_points(full_xyz[t, tgt], rep.K_front, rep.E_front[t])[0]
plt.figure(figsize=(4, 4))
plt.imshow(rep.front_rgb[t])
plt.plot(*tru, "o", mfc="none", mec="lime", ms=12, label="true")
plt.plot(*frozen_g[t, tgt, 2:4], "x", color="red", ms=10, label="frozen")
plt.plot(*bound_g[t, tgt, 2:4], "+", color="yellow", ms=12, label="bound")
plt.legend(fontsize=7)
plt.axis("off")
plt.title(f"target at t* = frame {t}")
plt.show()
print(
    f"error at t*: frozen {pxf[t]:.1f} px / {mf[t]:.3f} m, bound {pxb[t]:.1f} px / {mb[t]:.3f} m"
)
hid_t = np.nonzero(og[:, tgt, sg.F_HIDDEN] == 1)[0]
print(
    f"mean error while the target is hidden ({len(hid_t)} frames): "
    f"frozen {np.nanmean(pxf[hid_t]):.1f} px, bound {np.nanmean(pxb[hid_t]):.1f} px; "
    f"max frozen {np.nanmax(pxf[hid_t]):.1f} px, max bound {np.nanmax(pxb[hid_t]):.1f} px"
)
sg.pass_fail(
    "bound < frozen at t* (implementation check)",
    pxb[t] < pxf[t],
    (round(float(pxb[t]), 3), round(float(pxf[t]), 3)),
)

## Step 7. Graph-quality levels

**In:** the Q0 graph (visibility-gated oracle with the bound rule). **Out:** six versions of it, used only at test time:

| Level | What changes |
|---|---|
| Q0 | nothing |
| Q1_20, Q1_50 | the hidden active target (4.1) is shifted by a random pixel offset (sigma 20 or 50 px), drawn once per episode |
| Q2_25, Q2_50 | with probability 0.25 or 0.5 per episode, the hidden active target is put under a wrong container (one fixed wrong container per target) |
| Q3 | the perceived graph from Step 5 with the bound rule |

In [ ]:
# Step 7.1
show(sg.corrupt)
pbound, _ = sg.apply_rule(pg_n, "bound", grip_w, rep.ee_xyz, rep.K_front, rep.E_front)
levels = {
    q: sg.corrupt(
        bound_g,
        q,
        seed=0,
        active=active,
        targets=targets,
        scene=rep.scene,
        K=rep.K_front,
        E=rep.E_front,
        perceived_bound=pbound,
    )
    for q in sg.QUALITY_LEVELS
}

In [ ]:
# Step 7.2
fig, axes = plt.subplots(1, 6, figsize=(15, 3))
for a, (q, g) in zip(axes, levels.items()):
    a.imshow(rep.front_rgb[t_star_h5])
    a.plot(*tru, "o", mfc="none", mec="lime", ms=10)
    if np.all(np.isfinite(g[t_star_h5, tgt, 2:4])):
        a.plot(*g[t_star_h5, tgt, 2:4], "x", color="yellow", ms=10)
    a.set_title(q, fontsize=9)
    a.axis("off")
fig.suptitle("target as each level reports it at t* (green circle = true)")
fig.tight_layout()
fig.savefig(sg.FIG_DIR / "07_levels.png", dpi=120)
plt.show()

Each level below is a single change from Q0; the error it adds is printed per level.

In [ ]:
# Step 7.3
for q, g in levels.items():
    px, m = sg.target_error(g, full_xyz, tgt, rep.K_front, rep.E_front)
    print(f"{q:6s} target error at t*: {px[t_star_h5]:7.1f} px  {m[t_star_h5]:.3f} m")
for q in sg.QUALITY_LEVELS:
    if q.startswith(("Q1_", "Q2_")):
        q1, q2 = sg.draw_corruption(q, 0, targets, rep.scene)
        print(
            f"{q}: draw for seed 0 = "
            + (f"offset (du, dv) {np.round(q1, 1)} px" if q1 is not None else "")
            + (f"wrong container {q2}" if q2 is not None else "")
            + ("no corruption this episode" if q1 is None and q2 is None else "")
        )


In [ ]:
# Step 7.3b: Q2 really corrupts when its draw fires
q2_seed = next(sd for sd in range(100) if sg.draw_corruption("Q2_50", sd, targets, rep.scene)[1])
_, q2_map = sg.draw_corruption("Q2_50", q2_seed, targets, rep.scene)
wrong = q2_map[tgt]
g_q2 = sg.corrupt(
    bound_g, "Q2_50", q2_seed, active, targets, rep.scene, rep.K_front, rep.E_front
)
t = t_star_h5
px_q2, _ = sg.target_error(g_q2, full_xyz, active, rep.K_front, rep.E_front)
true_bin = rep.scene["cube_to_bin"][tgt]
print(f"first seed whose Q2_50 draw fires: {q2_seed}; wrong container for target {tgt}: {wrong}")
print(f"at t*: target reported at {np.round(g_q2[t, tgt, 2:4], 1)}, wrong container at "
      f"{np.round(bound_g[t, wrong, 2:4], 1)}, true container {true_bin} at {np.round(bound_g[t, true_bin, 2:4], 1)}")
print(f"target error at t*: {px_q2[t]:.1f} px (Q0: {pxb[t]:.1f} px)")
hid_t = np.nonzero(bound_g[:, tgt, sg.F_HIDDEN] == 1)[0]
moved = np.allclose(g_q2[hid_t, tgt, 2:4], bound_g[hid_t, wrong, 2:4], equal_nan=True)
sg.pass_fail(
    "Q2: hidden target takes the wrong container's position on every hidden frame",
    wrong != true_bin and moved and px_q2[t] > 20,
    (wrong, true_bin, round(float(px_q2[t]), 1)),
)


**How to read the output above.** = ? (after the run: Q0's error is the bound rule's own error from 6.5, not zero; Q1 errors scale with sigma; a Q2 draw either leaves the target alone or moves it to a wrong container, so on one episode it can show no error.)

The training cache below stores the Q0 graph for every cached episode, plus the DINOv2 vectors A3feat needs. Training never sees Q1 to Q3.

In [ ]:
# Step 7, cache: Q0 graphs (and A3feat features) for every cached training episode
for p in sorted((sg.CACHE_DIR / TASK).glob("ep*_replay.npz")):
    gp_ = p.with_name(p.name.replace("_replay", "_graph"))
    if gp_.exists():
        continue
    r_ = sg.load_replay(p)
    o_, _ = sg.oracle_graph(r_)
    o_[o_[..., sg.F_HIDDEN] == 1, sg.F_U : sg.F_Z + 1] = np.nan
    q0_, _ = sg.apply_rule(
        o_, "bound", r_.gripper_state.sum(1), r_.ee_xyz, r_.K_front, r_.E_front
    )
    np.savez_compressed(
        gp_, Q0=q0_, oracle_feats=sg.oracle_features(r_, proc, frames=list(r_.policy_frame))
    )
print("graph caches:", len(list((sg.CACHE_DIR / TASK).glob("ep*_graph.npz"))))

## Step 8. π0.5 and the three interfaces

**In:** the graph from Steps 4-7. **Out:** the same four fields per object (id, label, (u, v), hidden) delivered to π0.5 three ways, with the RoboMME recipe's settings.

π0.5, in four rungs:

1. **Tokens.** Each camera image becomes 256 tokens (SigLIP, 224 x 224 input, 14-pixel patches); the prompt becomes PaliGemma tokens. Every token is a vector of width d = 2048 (printed in 8.1).
2. **Prefix.** Image tokens of both cameras, then the prompt tokens `Task: ...;\nAction: `. Stock π0.5 also writes the robot state into the prompt as 256-bin numbers; RoboMME leaves proprioception out, and so does this notebook.
3. **Action expert.** A smaller transformer (300M parameters) attends to the prefix and turns noise into a chunk of 20 future joint commands by flow matching; 16 of them are executed before the next prediction.
4. **Where each interface enters.** A1 draws markers on the front image before SigLIP sees it; A2 appends the object list to the task text inside the prompt; A3 appends one learned token per object to the end of the prefix.

**What changes from the SmolVLA version** (details in `BUILD_NOTES.md`, "π0.5 version"):

| | SmolVLA (v1) | π0.5 (this notebook) |
|---|---|---|
| model | smolvla_base, ~0.45B | pi05_base, 3.2B (4.2B with SigLIP in LeRobot's count) |
| trained | LoRA, VLM text layers + action expert | LoRA, same scope; SigLIP frozen |
| action chunk | 50 predicted, 50 executed | 20 predicted, 16 executed |
| state | input token | not used (RoboMME: no proprioception) |
| normalisation | mean/std of lerobot/robomme (16 tasks) | q01/q99 of our training data |
| optimiser | AdamW 1e-3, clip 10 | AdamW 5e-5 (0.9, 0.95), warm-up, clip 1.0, EMA 0.999 |
| A3 tokens | before the state token | after the prompt tokens (no state token) |
| padded actions in the loss | masked | not masked (π0.5's loss has no pad mask) |

### 8.1 Inside π0.5

In [ ]:
# Step 8.1
print("perception worker stopped:", sg.close_perception_worker())  # frees ~8 GB before π0.5 loads
from lerobot.policies.pi05 import modeling_pi05, processor_pi05
from lerobot.policies.pi05.configuration_pi05 import PI05Config


def field_line(obj, prefix):
    """file:line of the first source line of obj starting with prefix."""
    src, start = inspect.getsourcelines(obj)
    i = next(i for i, l in enumerate(src) if l.strip().startswith(prefix))
    return f"{inspect.getsourcefile(obj)}:{start + i}  {src[i].strip()}"


print("π0.5 defaults (installed source):")
for f in ["chunk_size:", "n_action_steps:", "tokenizer_max_length:", "dtype:", "freeze_vision_encoder:", '"STATE":', '"ACTION":']:
    print("  ", field_line(PI05Config, f))
step = processor_pi05.Pi05PrepareStateTokenizerProcessorStep
print("   prompt with state:   ", field_line(step, 'full_prompt = f"Task: {cleaned_text}, State'))
print("   prompt without state:", field_line(step, 'full_prompt = f"Task: {cleaned_text};'))

stats = p5.quantile_stats(TRAIN_TASKS, CFG["train_episodes"])
policy, flow, pre, post, pcfg_policy, lmeta = p5.build_policy("cuda", stats=stats)
c = pcfg_policy
print("   prefix builder:      ", field_line(type(flow), "def embed_prefix"))
print("   training loss:       ", field_line(modeling_pi05.PI05Policy, "losses = losses[:, :, :original_action_dim]"))
print("\nvalues used here (pi05_policy.build_policy):")
for k in ["chunk_size", "n_action_steps", "tokenizer_max_length", "dtype", "freeze_vision_encoder", "gradient_checkpointing", "image_resolution"]:
    print(f"  {k} = {getattr(c, k)}")
print("  normalization_mapping =", {str(k): str(v) for k, v in c.normalization_mapping.items()})
print("  state in the prompt: no (Pi05PrepareStateTokenizerProcessorStep(include_state_in_prompt=False))")
print("input features:", list(c.input_features), "| output:", list(c.output_features))
D_MODEL = p5.d_model(flow)
print("prefix token width d =", D_MODEL, "| trainable LoRA parameters:", f"{sg.trainable_parameters(policy):,}")

In [ ]:
# Step 8.1 (prefix of one real sample)
import torch
from transformers import AutoTokenizer

tz = AutoTokenizer.from_pretrained(c.text_tokenizer_name)
smp = {
    "image": torch.from_numpy(rep.front_rgb[t_star_h5]).permute(2, 0, 1)[None].float() / 255,
    "wrist_image": torch.from_numpy(rep.wrist_rgb[t_star_h5]).permute(2, 0, 1)[None].float() / 255,
    "state": torch.from_numpy(
        np.r_[rep.joint_state[t_star_h5], rep.gripper_state[t_star_h5][:1]].astype(np.float32)
    )[None],
    "task": [rep.task_goal],
}
b = sg.prep(pre, smp)
inner = policy.get_base_model()
images, img_masks = inner._preprocess_images(b)
tokens, masks = b["observation.language.tokens"], b["observation.language.attention_mask"]
with torch.no_grad():
    embs, pad, att = flow.embed_prefix(images, img_masks, tokens, masks)
n_lang = tokens.shape[1]
print("prefix embeddings", tuple(embs.shape), "= [batch, tokens, d]")
print(f"  image tokens: {embs.shape[1] - n_lang} ({len(images)} cameras x {(embs.shape[1] - n_lang) // len(images)})")
print(f"  prompt tokens: {n_lang} slots, {int(masks.sum())} used")
print("  prompt as the model reads it:", repr(tz.decode(tokens[0][masks[0]])))
sg.pass_fail("no state in the prompt", "State" not in tz.decode(tokens[0][masks[0]]), "Task: ...;\\nAction:")

### 8.2 A1: markers drawn on the image

In [ ]:
# Step 8.2
show(sg.draw_markers)
from lerobot.policies.pi05.modeling_pi05 import resize_with_pad_torch

nodes = sg.nodes_at(bound_g, t_star_h5, rep.scene)
R = c.image_resolution[0]
fig, axes = plt.subplots(3, 3, figsize=(9, 9))
for i, size in enumerate([3, 6, 10]):
    marked = sg.draw_markers(rep.front_rgb[t_star_h5], nodes, size)
    x = torch.from_numpy(marked).float()[None] / 255  # [B, H, W, C], as π0.5's _preprocess_images
    pre_img = resize_with_pad_torch(x, R, R)[0].numpy()
    axes[i, 0].imshow(marked)
    axes[i, 1].imshow(pre_img)
    u, v = nodes[tgt]["u"], nodes[tgt]["v"]
    s2 = R / 256
    axes[i, 2].imshow(
        pre_img[max(0, int(v * s2) - 28) : int(v * s2) + 28, max(0, int(u * s2) - 28) : int(u * s2) + 28]
    )
    axes[i, 0].set_ylabel(f"size {size}")
for a, ttl in zip(axes[0], ["raw marked (256 px)", f"after π0.5 resize/pad ({R} px)", "zoom on target (4 x 4 SigLIP patches)"]):
    a.set_title(ttl, fontsize=8)
for a in axes.flat:
    a.set_xticks([])
    a.set_yticks([])
fig.tight_layout()
fig.savefig(sg.FIG_DIR / "08_markers_pi05.png", dpi=120)
plt.show()
MARKER_SIZE = 6
print("chosen marker size:", MARKER_SIZE, "= ? (confirm from the figure: smallest size whose ring and label stay distinct)")

### 8.3 A2: the graph as text inside π0.5's prompt

In [ ]:
# Step 8.3
show(sg.serialize)


def prompt_of(text):
    # the prompt Pi05PrepareStateTokenizerProcessorStep builds without state (see 8.1)
    return f"Task: {text.strip().replace('_', ' ').replace(chr(10), ' ')};\nAction: "


worst = (0, None)
for task_ in TRAIN_TASKS:
    for p in sorted((sg.CACHE_DIR / task_).glob("ep*_replay.npz"))[: CFG["train_episodes"]]:
        r_ = sg.load_replay(p, ["policy_frame"])
        g_ = sg.load_graph_cache(p.with_name(p.name.replace("_replay", "_graph")))["Q0"]
        for t_ in r_.policy_frame:
            s_ = prompt_of(sg.serialize(sg.nodes_at(g_, int(t_), r_.scene), r_.task_goal))
            worst = max(worst, (len(tz(s_)["input_ids"]), s_))
print("worst case prompt tokens:", worst[0], "| tokenizer_max_length:", c.tokenizer_max_length)
print(worst[1])
ids = tz(worst[1], truncation=True, max_length=c.tokenizer_max_length)["input_ids"]
print("decoded back (end):", repr(tz.decode(ids)[-100:]))
sg.pass_fail("no truncation at tokenizer_max_length", worst[0] <= c.tokenizer_max_length, (worst[0], c.tokenizer_max_length))

### 8.4 A3: graph tokens in π0.5's prefix

In [ ]:
# Step 8.4
show(sg.make_graph_tokenizer)
show(p5.GraphPrefix)
tok3 = sg.make_graph_tokenizer(3, D_MODEL).to("cuda")
gp = p5.GraphPrefix(flow, tok3)
x, lab, ids_, mask = sg.node_tensor(nodes, 256, 256, None)
gp.current = tuple(torch.from_numpy(a)[None].to("cuda") for a in (x, lab, ids_, mask))
with torch.no_grad():
    e2, p2, a2 = flow.embed_prefix(images, img_masks, tokens, masks)
print("prefix without graph", tuple(embs.shape), "| with graph", tuple(e2.shape),
      "| graph tokens", int(mask.sum()), "(+ padding", int((~mask).sum()), "masked)")
print("pad mask of the appended tokens:", p2[0, -len(mask):].int().tolist(), "| attention flags:", a2[0, -len(mask):].int().tolist())
policy.reset()
with torch.no_grad():
    act = policy.select_action(sg.prep(pre, smp))
print("action shape", tuple(act.shape), "| actions left in the queue:", len(inner._action_queue))
gp.current = None
flow.embed_prefix = gp.orig
sg.pass_fail("A3 forward runs, action shape unchanged", act.shape[-1] == 8 and len(inner._action_queue) == c.n_action_steps - 1, tuple(act.shape))

### 8.4c Label encoding

In [ ]:
# Step 8.4c
te = p5.text_embed_fn(flow, c)
for w in ["cube", "container"]:
    v = te(w)
    print(f"text label '{w}': vector {tuple(v.shape)}, norm {float(v.norm()):.2f}")
print("pilot uses label_mode='table'")

### 8.5 Equal capacity

In [ ]:
# Step 8.5
base = sg.trainable_parameters(policy)
rows = {"A0": base, "A1": base, "A2": base, "A3": base + sg.trainable_parameters(sg.make_graph_tokenizer(3, D_MODEL))}
for k, v in rows.items():
    print(f"{k:3s} trainable parameters {v:,}  (+{v - base:,} vs A0)")

A3 carries the graph tokenizer's extra parameters; this difference is reported with every result.

## Step 9. Training

**In:** cached replays and Q0 graphs of `TRAIN_TASKS`. **Out:** π0.5 checkpoints (LoRA adapters, EMA weights) for each interface.

One iteration as `sg.train(policy_name="pi05")` runs it: sample a policy step, its two images and the next 20 recorded actions; the interface changes the image (A1), the text (A2) or adds node tensors (A3); the preprocessor normalises actions with the quantiles of 9.1 and tokenises the prompt; π0.5's flow-matching loss is computed over the chunk; AdamW updates the LoRA adapters (and the A3 tokenizer); the EMA copy follows. Gradient accumulation (`accum_steps=2`) reaches RoboMME's batch 64 from batches of 32.

### 9.1 Normalisation statistics (q01/q99)

In [ ]:
# Step 9.1
show(p5.quantile_stats)
np.set_printoptions(precision=3, suppress=True, linewidth=140)
for k in ("action", "observation.state"):
    print(k)
    for s_ in ("q01", "q99", "mean", "std"):
        print(f"  {s_:4s}", np.array(stats[k][s_]))
print("stats file:", p5.stats_path(TRAIN_TASKS, CFG["train_episodes"]))
sg.pass_fail("q01/q99 for action and state", all(s_ in stats[k] for k in ("action", "observation.state") for s_ in ("q01", "q99")), list(stats))

### 9.2 A batch at chunk 20

In [ ]:
# Step 9.2
eps = [
    sg.load_training_episode(p)
    for task_ in TRAIN_TASKS
    for p in sorted((sg.CACHE_DIR / task_).glob("ep*_replay.npz"))[: CFG["train_episodes"]]
]
batch = sg.collate([sg.SGDataset(eps, "A3", chunk=p5.CHUNK_SIZE)[i] for i in range(4)])
for k_, v_ in batch.items():
    print(f"{k_:14s}", tuple(v_.shape) if hasattr(v_, "shape") else f"list of {len(v_)} strings")
print(f"{len(eps)} episodes, {sum(len(e[0].actions) for e in eps)} samples")

### 9.3 Smoke training: every interface, a few steps

In [ ]:
# Step 9.3
show(p5.make_optimizer)
import gc

for n_ in ["policy", "flow", "inner", "gp", "tok3", "te", "pre", "post", "b", "images", "img_masks",
           "tokens", "masks", "embs", "pad", "att", "e2", "p2", "a2", "act"]:
    globals().pop(n_, None)  # the Step 8 inspection model and every tensor or closure holding it
gc.collect()
torch.cuda.empty_cache()
smoke_info = {}
for iface in ["A0", "A1", "A2", "A3"]:
    smoke_info[iface] = sg.train(
        iface, TRAIN_TASKS, 0, CFG["steps"] or 10, sg.RUNS_DIR / f"pi05_smoke_{iface}",
        CFG["train_episodes"], batch_size=8, policy_name="pi05", log_every=5,
    )
    gc.collect()
    torch.cuda.empty_cache()
plt.figure(figsize=(5, 2.5))
for k_, v_ in smoke_info.items():
    plt.plot(v_["loss"], label=k_)
plt.xlabel("step")
plt.ylabel("flow-matching loss")
plt.legend()
plt.title("π0.5 smoke training (batch 8)")
plt.show()
from safetensors.torch import load_file

ok = all(np.isfinite(v_["loss"]).all() for v_ in smoke_info.values())
reloaded = load_file(str(sg.RUNS_DIR / "pi05_smoke_A2" / "policy" / "adapter_model.safetensors"))
sg.pass_fail("loss finite, checkpoint (EMA) reloads, stats saved",
             ok and len(reloaded) > 0 and (sg.RUNS_DIR / "pi05_smoke_A2" / "norm_stats.json").exists(),
             f"{len(reloaded)} adapter tensors")

### 9.4 Speed and memory at the pilot batch

In [ ]:
# Step 9.4: 25 steps at batch 32 (gradient checkpointing on), A0 and A2
PILOT_BATCH, PILOT_ACCUM, PILOT_STEPS = 32, 2, 80_000  # RoboMME: 80k steps at batch 64
speed = {}
for iface in ["A0", "A2"]:
    speed[iface] = sg.train(iface, TRAIN_TASKS, 0, 25, sg.RUNS_DIR / f"pi05_speed_{iface}",
                            CFG["train_episodes"], batch_size=PILOT_BATCH, policy_name="pi05", log_every=100)
    gc.collect()
    torch.cuda.empty_cache()
for k_, v_ in speed.items():
    print(f"{k_}: {v_['steps_per_s_steady']:.3f} steps/s after step 0, peak GPU memory {v_['max_mem_GB']:.1f} GB at batch {PILOT_BATCH}")
sps = min(v_["steps_per_s_steady"] for v_ in speed.values())
hours = p5.estimate_recipe_hours(1 / sps, PILOT_BATCH, PILOT_STEPS, PILOT_BATCH * PILOT_ACCUM)
print(f"RoboMME recipe ({PILOT_STEPS} steps, batch {PILOT_BATCH} x {PILOT_ACCUM}): ~{hours:.0f} h per arm on one GPU of this type")

**How to read the output above.** = ? (after the run: steps per second, peak memory, and the time per arm for the recipe.)

In [ ]:
# Step 9.5: pilot training (outside the notebook, one GPU per arm)
if RUN_MODE == "pilot":
    pilot_info = {}
    for iface in ["A0", "A1", "A2", "A3"]:
        p = sg.RUNS_DIR / f"pi05_{iface}_s0" / "train_info.json"
        if p.exists():
            pilot_info[iface] = json.loads(p.read_text())
            print(f"{iface}: {pilot_info[iface]['steps']} steps, {pilot_info[iface]['seconds'] / 3600:.1f} h")
        else:
            print(f"{iface}: not trained; run ./run_arms_parallel.sh --policy pi05 --interfaces A0,A1,A2,A3 "
                  f"--tasks permanence --steps {PILOT_STEPS} --batch {PILOT_BATCH} --accum {PILOT_ACCUM} --label pi05")
else:
    print("= ? (run in pilot mode)")

## Step 10. Evaluation

**In:** π0.5 checkpoints and the test split. **Out:** success rates and a rollout strip. Same loop as v1 (`sg.evaluate`): episodes capped at 1300 steps, the graph built from every frame returned at reset and updated online, 16 of every 20 predicted actions executed.

In [ ]:
# Step 10.1
info_ = json.loads((sg.RUNS_DIR / "pi05_smoke_A2" / "train_info.json").read_text())
print({k: info_[k] for k in ("policy", "chunk_size", "n_action_steps", "ema", "lr", "accum_steps")})
env = sg.make_env(TASK, "test", 0, max_steps=1300)
obs, info = env.reset()
env.close()
print("max_steps = 1300 | frames returned at reset (history video included):", len(obs["front_rgb_list"]))
sg.pass_fail("executes 16 of 20", (info_["chunk_size"], info_["n_action_steps"]) == (20, 16), (info_["chunk_size"], info_["n_action_steps"]))

In [ ]:
# Step 10.2: smoke evaluation, A2 on 2 episodes (rollout strip) and A3 on 1 (graph tokens online)
s_a2 = sg.evaluate(sg.RUNS_DIR / "pi05_smoke_A2", "A2", TASK, "Q0", 2, sg.RESULTS_DIR / "pi05_smoke_A2_Q0.json",
                   save_strip=sg.FIG_DIR / "10_rollout_pi05.png")
s_a3 = sg.evaluate(sg.RUNS_DIR / "pi05_smoke_A3", "A3", TASK, "Q0", 1, sg.RESULTS_DIR / "pi05_smoke_A3_Q0.json")
from IPython.display import Image

display(Image(str(sg.FIG_DIR / "10_rollout_pi05.png")))
for s_ in (s_a2, s_a3):
    print(s_["interface"], [(r_["episode"], r_["status"], r_["steps"]) for r_ in s_["results"]], f"{s_['seconds_per_episode']:.0f} s/episode")
sg.pass_fail("evaluation runs for A2 and A3", len(s_a2["results"]) == 2 and len(s_a3["results"]) == 1, "3 episodes")

In [ ]:
# Step 10.3: pilot evaluation, every arm on 50 test episodes of each permanence task at Q0
if RUN_MODE == "pilot":
    motion = sg.load_motion("VideoUnmaskSwap")
    for iface in ["A0", "A1", "A2", "A3"]:
        for task_ in TRAIN_TASKS:
            out_ = sg.RESULTS_DIR / f"pi05_{iface}_{task_}_Q0.json"
            r_ = json.loads(out_.read_text()) if out_.exists() else sg.evaluate(
                sg.RUNS_DIR / f"pi05_{iface}_s0", iface, task_, "Q0", CFG["eval_episodes"], out_)
            for name, sp in sg.success_by_split(r_, motion if task_ == "VideoUnmaskSwap" else None).items():
                print(f"{iface} {task_:16s} {name:14s} {sp['k']}/{sp['n']} = {sp['rate']:.3f}, 95% Wilson {np.round(sp['wilson95'], 3)}")
else:
    print("= ? (run in pilot mode)")

**How to read the output above.** = ? (after the pilot run.)

In [ ]:
# Step 10.4: grid size and cost (not run)
n_arms, n_seeds = 4, 3
print(f"π0.5 grid: {n_arms} interfaces x {n_seeds} seeds = {n_arms * n_seeds} training runs of ~{hours:.0f} h "
      f"= ~{n_arms * n_seeds * hours:.0f} GPU h at the speed measured in 9.4")
print("launch per seed: ./run_arms_parallel.sh --policy pi05 --interfaces A0,A1,A2,A3 --tasks permanence "
      f"--steps {PILOT_STEPS} --batch {PILOT_BATCH} --accum {PILOT_ACCUM} --seed <s> --label pi05")

## References

- RoboMME: Dai et al., *RoboMME: Benchmarking and Understanding Memory for Robotic Generalist Policies*, ICML 2026, https://arxiv.org/abs/2603.04639; code https://github.com/RoboMME/robomme_benchmark (read at commit 53673b3: `doc/env_format.md`, `doc/h5_data_format.md`, `scripts/dataset_replay.py`, `src/robomme/robomme_env/VideoUnmaskSwap.py`).
- SmolVLA and LeRobot: https://github.com/huggingface/lerobot (read at commit 5831498: `policies/smolvla/*`, `envs/robomme.py`, `scripts/lerobot_train.py`, `docs/source/robomme.mdx`, `docs/source/peft_training.mdx`).
- Scene-graph pipeline: Qian et al., *Expanding Spatial and Temporal Context for Robotic Imitation Learning With Scene Graphs*, CVPR 2026, https://arxiv.org/abs/2606.01072; this repo's `post_process_scene_cutie.py`, `extract_part_masks_cutie.py`, `dinov2_encoder.py`.
- π0.5: Physical Intelligence, *π0.5: a Vision-Language-Action Model with Open-World Generalization*, 2025; LeRobot port `lerobot/policies/pi05` (read at commit 5831498: `configuration_pi05.py`, `modeling_pi05.py`, `processor_pi05.py`), weights `lerobot/pi05_base`, tokenizer `google/paligemma-3b-pt-224`.
- RoboMME training recipe: arXiv 2603.04639 v3, Appendix B.2.3 and Table 6; §5.1 (multi-task training, chunk 20 / execute 16).
